In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import warnings
from xgboost import XGBRegressor
import optuna

In [25]:
# 경고 메시지 무시
warnings.filterwarnings('ignore', category=FutureWarning)

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('data.csv', encoding='utf-8')

In [26]:
# 5대 리그 필터링
top_5_leagues = ['Bundesliga', 'LaLiga', 'Premier League', 'Serie A', 'Ligue 1']
df = df[df['리그'].isin(top_5_leagues)].copy()

# 기본 전처리
df['TM_시장가치_EUR'] = pd.to_numeric(df['TM_시장가치_EUR'], errors='coerce')
df.dropna(subset=['TM_시장가치_EUR', '포지션'], inplace=True)
df = df[df['TM_시장가치_EUR'] > 0]
df = df[df['포지션'] != '골키퍼'].copy()


# 2. 피쳐 엔지니어링 (Feature Engineering)
df.sort_values(by=['name_key', '시즌'], inplace=True)
df['나이_제곱'] = df['시즌종료시점만나이'] ** 2

stats_for_change = ['득점', '어시스트', '기회 창출', '슛', '태클 성공', '가로채기', '터치', '성공한 패스']

for stat in stats_for_change:
    df[f'전시즌_{stat}'] = df.groupby('name_key')[stat].shift(1).fillna(0)
    df[f'{stat}_변화량'] = df[stat] - df[f'전시즌_{stat}']

df['나이x득점_변화량'] = (df['시즌종료시점만나이'] * df['득점_변화량'])
df['슛_대비_득점율'] = np.divide(df['득점'], df['슛'], out=np.zeros_like(df['득점'], dtype=float), where=df['슛']!=0)

df['전시즌_시장가치_EUR'] = df.groupby('name_key')['TM_시장가치_EUR'].shift(1).fillna(0)
df['log_전시즌_시장가치'] = np.log1p(df['전시즌_시장가치_EUR'])
df['log_현재_시장가치'] = np.log1p(df['TM_시장가치_EUR'])
df['시장가치_변화량'] = df['TM_시장가치_EUR'] - df['전시즌_시장가치_EUR']

In [27]:
# 3. 포지션 그룹핑
def group_position_detailed(pos):
    pos_str = str(pos)
    if '중앙 수비수' in pos_str: return '중앙 수비수'
    if '왼쪽 수비수' in pos_str or '오른쪽 수비수' in pos_str: return '측면 수비수'
    if '수비형 미드필더' in pos_str: return '수비형 미드필더'
    if '공격형 미드필더' in pos_str: return '공격형 미드필더'
    if '중앙 미드필더' in pos_str: return '중앙 미드필더'
    if '윙어' in pos_str or '왼쪽 미드필더' in pos_str or '오른쪽 미드필더' in pos_str: return '윙어'
    if '스트라이커' in pos_str: return '스트라이커'
    return '기타'
df['포지션_그룹'] = df['포지션'].apply(group_position_detailed)
df = df[df['포지션_그룹'] != '기타']

In [28]:
best_params_per_position = {
    '공격형 미드필더': {'n_estimators': 1300, 'learning_rate': 0.043046501045109875, 'max_depth': 12, 'subsample': 0.7146481658848329, 'colsample_bytree': 0.5868849207152588, 'reg_alpha': 0.9632756396268152, 'reg_lambda': 0.0003821877960217409},
    '수비형 미드필더': {'n_estimators': 1200, 'learning_rate': 0.04380824140046239, 'max_depth': 17, 'subsample': 0.3847060064183181, 'colsample_bytree': 0.4127967809883891, 'reg_alpha': 0.014734131067062008, 'reg_lambda': 1.9694184672233443e-05},
    '스트라이커': {'n_estimators': 1100, 'learning_rate': 0.0376632474408951, 'max_depth': 15, 'subsample': 0.6268242620148625, 'colsample_bytree': 0.2985317568222804, 'reg_alpha': 1.3203548526180334e-06, 'reg_lambda': 0.0008133122545807272},
    '윙어': {'n_estimators': 1500, 'learning_rate': 0.07284, 'max_depth': 13, 'subsample': 0.9931, 'colsample_bytree': 0.2543, 'reg_alpha': 7.460e-06, 'reg_lambda': 5.387e-06}, # 2차 결과
    '중앙 미드필더': {'n_estimators': 1600, 'learning_rate': 0.07079413461238956, 'max_depth': 8, 'subsample': 0.8327918966197049, 'colsample_bytree': 0.580856531512097, 'reg_alpha': 3.0326869119156524e-05, 'reg_lambda': 0.0844581233820778}, # 2차 결과
    '중앙 수비수': {'n_estimators': 1500, 'learning_rate': 0.028101536180308466, 'max_depth': 9, 'subsample': 0.39591894623275053, 'colsample_bytree': 0.1690295563927809, 'reg_alpha': 0.8214238173442158, 'reg_lambda': 0.7589281734056073},
    '측면 수비수': {'n_estimators': 900, 'learning_rate': 0.027658997275438843, 'max_depth': 8, 'subsample': 0.9631908444945139, 'colsample_bytree': 0.37477240790734695, 'reg_alpha': 6.207178940305591e-07, 'reg_lambda': 6.860795293402658e-08}
}

In [29]:
position_groups = sorted(df['포지션_그룹'].unique())
for group in position_groups:
    print("\n" + "="*70)
    print(f"포지션 그룹 최종 분석 (Champion Model): ✨ {group} ✨")
    print("="*70)

    group_df = df[df['포지션_그룹'] == group].copy()
    if len(group_df) < 20: continue

    y = group_df['log_현재_시장가치']
    
    cols_to_drop = [
        '리그', 'TM_시장가치_EUR', '전시즌_시장가치_EUR', '시장가치_변화량', 'log_현재_시장가치',
        '선수명', '생년월일', 'source_file', 'name_key', '시즌', '시즌코드', 
        '포지션', '포지션_그룹'
    ] + [f'전시즌_{stat}' for stat in stats_for_change]
    X = group_df.drop(columns=cols_to_drop).fillna(0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # ---- [핵심 변경점 2: Optuna 대신, 정의된 최적 파라미터 사용] ----
    print(f"'{group}' 포지션의 최종 챔피언 모델을 학습합니다...")
    
    # 해당 그룹의 최적 파라미터를 딕셔너리에서 가져오기
    champion_params = best_params_per_position[group]
    
    final_model = XGBRegressor(
        objective='reg:squarederror',
        **champion_params, # 저장된 최적의 파라미터 적용
        random_state=42,
        n_jobs=-1
    )
    
    final_model.fit(X_train_scaled, y_train, verbose=False)

    # --- 최종 모델 평가 및 오차 분석 ---
    log_y_pred = final_model.predict(X_test_scaled)
    y_pred = np.expm1(log_y_pred)
    y_test_orig = np.expm1(y_test)
    
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
    r2 = r2_score(y_test, log_y_pred) 
    
    print(f"\n[📊 **챔피언 모델** 성능 평가 - {group}]")
    print(f"  - 평균 예측 오차 (RMSE): {rmse:,.0f} EUR")
    print(f"  - 모델 설명력 (R-squared): {r2:.2%}")

    # (피쳐 중요도, 오차 분석 등 나머지 코드는 이전과 동일...)
    # 피쳐 중요도 분석
    importances = final_model.feature_importances_
    feature_names = X.columns
    imp_df = pd.DataFrame(sorted(zip(importances, feature_names), reverse=True), columns=['Importance', 'Feature']).head(15)

    print(f"\n[📈 'log(시장가치)'에 영향을 미치는 주요 피쳐 - {group}]")
    pd.set_option('display.float_format', '{:.4f}'.format)
    imp_df_display = imp_df.copy()
    imp_df_display.index = np.arange(1, len(imp_df_display) + 1)
    print(imp_df_display)
    
    # plt.figure(figsize=(10, 6))
    # sns.barplot(x='Importance', y='Feature', data=imp_df, palette='viridis')
    # plt.title(f"'{group}'의 log(시장가치) 주요 피쳐 (Champion Model)", fontsize=15)
    # plt.xlabel('중요도')
    # plt.ylabel('스탯')
    # plt.tight_layout()
    # plt.show()

    # 오차 심층 분석
    results_df = X_test.copy()
    results_df['선수명'] = df.loc[X_test.index, '선수명'] 
    results_df['전시즌_시장가치_EUR'] = df.loc[X_test.index, '전시즌_시장가치_EUR']
    results_df['실제_시장가치_EUR'] = y_test_orig 
    results_df['예측_시장가치_EUR'] = pd.Series(y_pred, index=X_test.index)
    results_df['예측_오차_EUR'] = results_df['예측_시장가치_EUR'] - results_df['실제_시장가치_EUR']
    
    analysis_cols = ['선수명', '시즌종료시점만나이', '전시즌_시장가치_EUR', 
                     '실제_시장가치_EUR', '예측_시장가치_EUR', '예측_오차_EUR']
    error_analysis_df = results_df[analysis_cols].copy()

    print("\n" + "*"*30)
    print(f"  예측 오차 심층 분석 - {group}")
    print("*"*30)
    print("\n[⬆️ 모델이 가장 과대평가한 선수 TOP 5]")
    print(error_analysis_df.sort_values(by='예측_오차_EUR', ascending=False).head(5))
    print("\n[⬇️ 모델이 가장 과소평가한 선수 TOP 5]")
    print(error_analysis_df.sort_values(by='예측_오차_EUR', ascending=True).head(5))
    print("\n[🎯 모델이 가장 정확하게 예측한 선수 TOP 5]")
    most_accurate = error_analysis_df.copy()
    most_accurate['절대오차'] = most_accurate['예측_오차_EUR'].abs()
    print(most_accurate.sort_values(by='절대오차', ascending=True).head(5).drop(columns='절대오차'))
    print("-" * 70)


포지션 그룹 최종 분석 (Champion Model): ✨ 공격형 미드필더 ✨
'공격형 미드필더' 포지션의 최종 챔피언 모델을 학습합니다...

[📊 **챔피언 모델** 성능 평가 - 공격형 미드필더]
  - 평균 예측 오차 (RMSE): 9,697,806 EUR
  - 모델 설명력 (R-squared): 79.43%

[📈 'log(시장가치)'에 영향을 미치는 주요 피쳐 - 공격형 미드필더]
    Importance          Feature
1       0.1277  xG 유효 슈팅 (xGOT)
2       0.0914     log_전시즌_시장가치
3       0.0903     예상 어시스트 (xA)
4       0.0866        예상 골 (xG)
5       0.0365           성공한 패스
6       0.0300                슛
7       0.0274     공중 볼 경합 성공 %
8       0.0261           패스 정확도
9       0.0244               터치
10      0.0231        PK 제외한 xG
11      0.0202             가로채기
12      0.0195       성공한 패스_변화량
13      0.0194            나이_제곱
14      0.0182        시즌종료시점만나이
15      0.0175         롱 패스 정확도

******************************
  예측 오차 심층 분석 - 공격형 미드필더
******************************

[⬆️ 모델이 가장 과대평가한 선수 TOP 5]
                    선수명  시즌종료시점만나이  전시즌_시장가치_EUR   실제_시장가치_EUR   예측_시장가치_EUR  \
4119         joao félix         24 67500000.0000 33333333.0000 676454